### Compute TCR Similarity Based on K-mer and Physicochemical Properties

In [ ]:
import pandas as pd
import numpy as np
import os

script_dir = os.path.dirname(os.path.abspath("2a-similarity_tcr.ipynb"))

In [ ]:
# Prepare TCR sequences

data_path = os.path.join(script_dir, "../data", "unique_TCRs.tsv")
unique_tcrs = pd.read_csv(data_path, sep='\t')

alpha_seqs = unique_tcrs['TCR_alpha_CDR3'].astype(str).tolist()
beta_seqs = unique_tcrs['TCR_beta_CDR3'].astype(str).tolist()
n = len(unique_tcrs)

print(f"Loaded {n} unique TCR sequences")

# define paramters
beta_weight = 0.7
kmer_weight = 0.5

In [ ]:
# Define physicochemical properties using AAindex (5 properties)
def load_aaindex_properties():
    # 5 AAindex physicochemical properties, normalized to zero mean / unit variance
    # AAindex entries
    aaindex_data = {
        # KYTJ820101: Hydropathy index (Kyte-Doolittle, 1982)
        'hydrophobicity': {
            'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
            'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
            'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
            'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
        },
        # FAUJ880103: Normalized net charge (Fauchere et al., 1988)
        'charge': {
            'A': 0.0, 'R': 1.0, 'N': 0.0, 'D': -1.0, 'C': 0.0,
            'Q': 0.0, 'E': -1.0, 'G': 0.0, 'H': 0.5, 'I': 0.0,
            'L': 0.0, 'K': 1.0, 'M': 0.0, 'F': 0.0, 'P': 0.0,
            'S': 0.0, 'T': 0.0, 'W': 0.0, 'Y': 0.0, 'V': 0.0
        },
        # GRAR740102: Polarity (Grantham, 1974)
        'polarity': {
            'A': 8.1, 'R': 10.5, 'N': 11.6, 'D': 13.0, 'C': 5.5,
            'Q': 10.5, 'E': 12.3, 'G': 9.0, 'H': 10.4, 'I': 5.2,
            'L': 4.9, 'K': 11.3, 'M': 5.7, 'F': 5.2, 'P': 8.0,
            'S': 9.2, 'T': 8.6, 'W': 5.4, 'Y': 6.2, 'V': 5.9
        },
        # FASG760101: Molecular weight (Fasman, 1976)
        'volume': {
            'A': 89.0, 'R': 174.0, 'N': 132.0, 'D': 133.0, 'C': 121.0,
            'Q': 146.0, 'E': 147.0, 'G': 75.0, 'H': 155.0, 'I': 131.0,
            'L': 131.0, 'K': 146.0, 'M': 149.0, 'F': 165.0, 'P': 115.0,
            'S': 105.0, 'T': 119.0, 'W': 204.0, 'Y': 181.0, 'V': 117.0
        },
        # ZIMJ680104: Isoelectric point (Zimmerman et al., 1968)
        'isoelectric_point': {
            'A': 6.00, 'R': 10.76, 'N': 5.41, 'D': 2.77, 'C': 5.02,
            'Q': 5.65, 'E': 3.22, 'G': 5.97, 'H': 7.59, 'I': 6.02,
            'L': 5.98, 'K': 9.74, 'M': 5.74, 'F': 5.48, 'P': 6.30,
            'S': 5.68, 'T': 5.64, 'W': 5.89, 'Y': 5.66, 'V': 5.96
        },
    }
    
    # Normalize each property to zero mean and unit variance
    for prop_name, prop_dict in aaindex_data.items():
        values = np.array(list(prop_dict.values()))
        mean_val = np.mean(values)
        std_val = np.std(values)
        
        for aa in prop_dict:
            prop_dict[aa] = (prop_dict[aa] - mean_val) / std_val
    
    # Create combined property vectors
    amino_acids = list('ARNDCQEGHILKMFPSTWYV')
    properties = list(aaindex_data.keys())
    
    physicochemical_properties = {}
    for aa in amino_acids:
        physicochemical_properties[aa] = [
            aaindex_data[prop][aa] for prop in properties
        ]
    
    return physicochemical_properties, properties

physicochemical_properties, property_names = load_aaindex_properties()

print("Using 5 AAindex properties:")
for i, prop in enumerate(property_names):
    print(f"  {i+1}. {prop}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

def compute_kmer_vectors(sequences, k=3):
    # Precompute k-mer count vectors for all sequences
    all_kmers = set()
    for seq in sequences:
        all_kmers.update([seq[i:i+k] for i in range(len(seq) - k + 1)])
    
    kmer_to_idx = {kmer: idx for idx, kmer in enumerate(sorted(all_kmers))}
    print(f"  Found {len(kmer_to_idx)} unique k-mers")
    
    print(f"  Vectorizing sequences...")
    vectors = np.zeros((len(sequences), len(kmer_to_idx)))
    
    for i, seq in enumerate(sequences):
        kmers = [seq[j:j+k] for j in range(len(seq) - k + 1)]
        counter = Counter(kmers)
        for kmer, count in counter.items():
            vectors[i, kmer_to_idx[kmer]] = count
    
    print(f"  Normalizing vectors...")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1  # avoid division by zero
    vectors = vectors / norms
    
    return vectors

# Compute all pairwise k-mer similarities at once
kmer_sim_alpha = cosine_similarity(np.concatenate([compute_kmer_vectors(alpha_seqs, k=2), compute_kmer_vectors(alpha_seqs, k=3)], axis=1))
kmer_sim_beta = cosine_similarity(np.concatenate([compute_kmer_vectors(beta_seqs, k=2), compute_kmer_vectors(beta_seqs, k=3)], axis=1))

# Combine alpha and beta k-mer similarities  alpha, beta
print("  Combining alpha and beta k-mer similarities...")
sim_matrix_kmer = (1-beta_weight) * kmer_sim_alpha + beta_weight * kmer_sim_beta

# Compute all pairwise physicochemical similarities at once
def sequence_to_property_vector(seq, properties_dict):

    if not seq or seq == 'nan' or pd.isna(seq):
        print("  Warning: Empty or NaN sequence encountered.")
        return np.zeros(len(property_names))
    
    vectors = []
    for aa in seq:
        if aa in properties_dict:
            vectors.append(properties_dict[aa])
    
    if not vectors:
        return np.zeros(len(property_names))
    
    return np.mean(vectors, axis=0)

# Convert all sequences to property vectors for physicochemical similarity
alpha_prop_vectors = np.array([sequence_to_property_vector(s, physicochemical_properties) 
                                for s in alpha_seqs])
beta_prop_vectors = np.array([sequence_to_property_vector(s, physicochemical_properties) 
                               for s in beta_seqs])

print(f"Alpha property vectors shape: {alpha_prop_vectors.shape}")
print(f"Beta property vectors shape: {beta_prop_vectors.shape}")

physchem_sim_alpha = cosine_similarity(alpha_prop_vectors)
physchem_sim_beta = cosine_similarity(beta_prop_vectors)

# Map cosine similarity from [-1,1] to [0,1] and combine alpha, beta
physchem_sim_alpha_norm = (physchem_sim_alpha + 1.0) / 2.0
physchem_sim_beta_norm = (physchem_sim_beta + 1.0) / 2.0
sim_matrix_physchem = (1-beta_weight) * physchem_sim_alpha_norm + beta_weight * physchem_sim_beta_norm

# Standardize similarity matrices based on empirical mean/std
sim_matrix_kmer = (sim_matrix_kmer - np.mean(sim_matrix_kmer[np.triu_indices(n, k=1)])) / np.std(sim_matrix_kmer[np.triu_indices(n, k=1)])
sim_matrix_physchem = (sim_matrix_physchem - np.mean(sim_matrix_physchem[np.triu_indices(n, k=1)])) / np.std(sim_matrix_physchem[np.triu_indices(n, k=1)])

# Final combined similarity: kmer_weight * S_kmer + (1-kmer_weight) * S_AA
sim_matrix_combined = kmer_weight * sim_matrix_kmer + (1-kmer_weight) * sim_matrix_physchem

# Clamp to [0,1]
sim_matrix_combined = np.clip(sim_matrix_combined, 0.0, 1.0)

# Get number of sequences
n = len(alpha_seqs)

print(f"\nCombined similarity matrix shape: {sim_matrix_combined.shape}")
print(f"Combined similarity range: [{sim_matrix_combined.min():.3f}, {sim_matrix_combined.max():.3f}]")
print(f"Mean combined similarity (off-diagonal): {np.mean(sim_matrix_combined[np.triu_indices(n, k=1)]):.3f}")
print(f"\nK-mer similarity - Mean: {np.mean(sim_matrix_kmer[np.triu_indices(n, k=1)]):.3f}, Std: {np.std(sim_matrix_kmer[np.triu_indices(n, k=1)]):.3f}")
print(f"Physicochemical similarity - Mean: {np.mean(sim_matrix_physchem[np.triu_indices(n, k=1)]):.3f}, Std: {np.std(sim_matrix_physchem[np.triu_indices(n, k=1)]):.3f}")

In [ ]:
# Save the similarity matrices
from pathlib import Path

out_dir = Path("../data")
out_dir.mkdir(exist_ok=True)
np.save(out_dir / "TCR_similarity_combined.npy", sim_matrix_combined)
np.save(out_dir / "TCR_similarity_kmer.npy", sim_matrix_kmer)
np.save(out_dir / "TCR_similarity_physchem.npy", sim_matrix_physchem)